# 🌬️ Respiration Rate from a Thermal Video

### A self-contained Colab on sensing breathing from airway temperature
**ACM Summer School on AI for Health — Hands-on Notebook**
*Built around the JoulesEye project · Prepared by Nipun Batra's group*

---

**Respiration rate is the neglected vital sign.** Heart rate, blood pressure and
oxygen saturation are measured everywhere, but breathing rate (breaths per
minute, "brpm") is rarely tracked at home or even in the clinic — despite being
one of the earliest and most sensitive markers of deterioration:

- Elevated stress / anxiety: **> 25 brpm**
- Pneumonia: **> 30 brpm**
- Impending cardiopulmonary arrest: **> 27 brpm**

**JoulesEye** senses respiration rate from a **thermal camera** pointed at the
face. As you breathe, air flows past the nostrils and the airway region changes
temperature: the **exhale is warm**, the **inhale is cool**. Averaging the
temperature inside a small nostril region of interest (ROI) over time therefore
produces a 1-D signal that oscillates **at the breathing rate** — exactly the
kind of periodic signal a standard pipeline can lock onto.

In this notebook you will:

1. **Build intuition** by synthesising a respiration signal at a *known* rate and
   recovering it — so you trust the pipeline before trusting pixels.
2. **Exercise the real video code path** on a programmatically generated thermal
   "nostril" video (so the notebook runs end-to-end with no upload).
3. **Go from respiration to energy expenditure (EE)** using the Fick principle,
   and reproduce JoulesEye's headline result: knowing breathing rate makes
   calorie estimates far more accurate than a heart-rate-only smartwatch.

> ⚠️ **This is an educational demo, not a medical device.** Nothing here is
> validated for diagnosis or clinical decisions.

## 0. The physics in one paragraph

A thermal camera measures the infrared radiation (temperature) of every pixel.
Near the nostrils, the air you exhale is close to body temperature (warm) while
the air you inhale is closer to ambient (cool). Across a breathing cycle the
**mean temperature of a small nostril ROI rises and falls** once per breath, so
the per-frame ROI mean is a respiration waveform sampled at the camera frame
rate. Breathing lives at roughly **0.1–0.6 Hz (6–36 breaths/min)**, well below
the heart-rate band, so a band-pass filter cleanly isolates it.

The whole algorithm is therefore:

```
thermal frames ──▶ track nostril ROI ──▶ per-frame mean temperature ──▶ s(t)
   s(t) ──▶ detrend ──▶ band-pass 0.1–0.6 Hz ──▶ FFT / peak-detect ──▶ rate
```

The one hard part during **exercise** is keeping the ROI on the nostrils while
the head moves. Classic optical flow struggles on low-texture thermal imagery,
so JoulesEye learns a dedicated **ROI localiser**. We illustrate this failure
mode in the bonus section.

## 1. Setup
Everything below is pre-installed on Colab. The `pip install` is a portable
no-op there, so the notebook also runs on any local machine with OpenCV.

In [ ]:
# Colab already has these; the install is just for portability.
try:
    import cv2  # noqa
except ImportError:
    import sys, subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "opencv-python-headless"], check=True)

import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks, welch, detrend

np.set_printoptions(suppress=True)
plt.rcParams.update({"figure.figsize": (11, 3.2), "axes.grid": True,
                     "grid.alpha": 0.3, "font.size": 11})

IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else False
print("Running in Colab:", IN_COLAB)

## 2. The signal-processing pipeline (define once, reuse everywhere)

These four functions are the entire respiration-rate estimator. We define them
up front and then apply the *exact same* functions to (a) a synthetic 1-D signal
and (b) a synthetic thermal video — so there is no hidden magic. Note the band
is now **0.1–0.6 Hz** (breathing), not the heart-rate band.

In [ ]:
def bandpass(x, fs, lo=0.1, hi=0.6, order=3):
    """Zero-phase Butterworth band-pass for the BREATHING band. Keeps
    respiration frequencies (lo..hi Hz, i.e. 6..36 breaths/min) and removes slow
    thermal baseline drift plus high-frequency sensor noise."""
    nyq = 0.5 * fs
    hi = min(hi, 0.99 * nyq)               # guard: fs must support hi
    b, a = butter(order, [lo / nyq, hi / nyq], btype="band")
    return filtfilt(b, a, x)


def rr_from_fft(x, fs, lo=0.1, hi=0.6):
    """Respiration rate (breaths/min) = frequency of the largest spectral peak
    inside the breathing band [lo, hi] Hz."""
    x = detrend(np.asarray(x, float))
    n = len(x)
    win = np.hanning(n)
    freqs = np.fft.rfftfreq(n, d=1.0 / fs)
    power = np.abs(np.fft.rfft(x * win)) ** 2
    band = (freqs >= lo) & (freqs <= hi)
    f_peak = freqs[band][np.argmax(power[band])]
    return f_peak * 60.0, freqs, power


def rr_from_peaks(x, fs, lo=0.1, hi=0.6):
    """Respiration rate from time-domain breath detection. Also returns the peak
    indices, the filtered signal, and inter-breath intervals (in seconds)."""
    xf = bandpass(detrend(np.asarray(x, float)), fs, lo, hi)
    min_dist = int(fs / hi)                # breaths cannot be closer than 1/hi s
    peaks, _ = find_peaks(xf, distance=max(1, min_dist),
                          prominence=0.3 * np.std(xf))
    ibi = np.diff(peaks) / fs              # seconds between successive breaths
    rr = 60.0 / np.median(ibi) if len(ibi) else np.nan
    return rr, peaks, xf, ibi


def signal_quality(x, fs, lo=0.1, hi=0.6):
    """Crude SNR: fraction of total spectral power that falls inside the
    breathing band. Values above ~0.2 usually indicate a usable trace."""
    f, P = welch(detrend(np.asarray(x, float)), fs=fs,
                 nperseg=min(len(x), int(fs * 16)))
    band = (f >= lo) & (f <= hi)
    return float(P[band].sum() / P.sum())

## 3. Sanity check on a *synthetic* respiration signal

Before touching video, we manufacture a respiration-like waveform at a **known**
rate (15 breaths/min), add a slow thermal baseline drift and sensor noise, and
confirm the pipeline recovers the truth. Real breathing is not a pure sine — the
exhale is a little sharper than the inhale — so we add a second harmonic.

In [ ]:
def synth_resp(rr_brpm=15.0, fs=30.0, dur=60.0, noise=0.15, drift=0.6, seed=0):
    """Synthesise a 1-D respiration trace at a KNOWN rate: a breathing
    oscillation (sharper exhale via a 2nd harmonic) on top of a slow thermal
    baseline drift, plus white sensor noise. Returns (t, signal)."""
    rng = np.random.default_rng(seed)
    t = np.arange(0, dur, 1.0 / fs)
    f = rr_brpm / 60.0
    ph = 2 * np.pi * f * t
    breath = np.sin(ph) + 0.25 * np.sin(2 * ph + 0.5)
    baseline = drift * np.sin(2 * np.pi * 0.01 * t + 0.3)    # very slow drift
    sig = breath + baseline + noise * rng.standard_normal(t.size)
    return t, sig

TRUE_RR = 15.0
FS_SYNTH = 30.0
t, s = synth_resp(rr_brpm=TRUE_RR, fs=FS_SYNTH, dur=60.0, noise=0.15)

rr_fft, freqs, power = rr_from_fft(s, FS_SYNTH)
rr_pk, peaks, s_filt, ibi = rr_from_peaks(s, FS_SYNTH)
print(f"True respiration rate : {TRUE_RR:.1f} breaths/min")
print(f"FFT estimate          : {rr_fft:.1f} breaths/min")
print(f"Peak estimate         : {rr_pk:.1f} breaths/min")
print(f"Signal quality        : {signal_quality(s, FS_SYNTH):.2f}")
print(f"Breaths detected      : {len(peaks)} over {len(s) / FS_SYNTH:.0f} s")

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 8))

ax[0].plot(t, s, lw=1, color="#888")
ax[0].set(title="Raw synthetic respiration trace (baseline drift + noise)",
          xlabel="time (s)", ylabel="temperature proxy (a.u.)")

ax[1].plot(t, s_filt, lw=1.3, color="#1f77b4")
ax[1].plot(t[peaks], s_filt[peaks], "rv", ms=7, label="detected breaths")
ax[1].set(title=f"Band-passed 0.1-0.6 Hz + breath detection  →  {rr_pk:.1f} brpm",
          xlabel="time (s)", ylabel="a.u.")
ax[1].legend(loc="upper right")

brpm_axis = freqs * 60
m = (brpm_axis >= 0) & (brpm_axis <= 60)
ax[2].plot(brpm_axis[m], power[m] / power[m].max(), color="#2ca02c")
ax[2].axvline(rr_fft, color="r", ls="--", label=f"peak = {rr_fft:.1f} brpm")
ax[2].set(title="Spectrum (FFT)", xlabel="respiration rate (breaths/min)",
          ylabel="norm. power")
ax[2].legend()
plt.tight_layout(); plt.show()

✅ Both estimators land within ~1 breath/min of the truth. The pipeline works —
now let's feed it thermal pixels.

## 4. From thermal video to a respiration signal

To run end-to-end **without any upload**, we synthesise a short thermal video: a
cool face background with a warm **nostril blob** at the centre whose intensity
oscillates at a known breathing rate (plus sensor noise). Then
`extract_respiration_from_video` reads it frame-by-frame, averages a small
central ROI, and hands the 1-D signal to the *same* pipeline. Recovering the
injected rate proves the **video ──▶ respiration rate** path end-to-end.

In [ ]:
def make_thermal_video(path, rr_brpm=18.0, fps=30.0, dur=30.0, size=(160, 120),
                       warm=120.0, amp=18.0, noise=3.0, seed=1):
    """Write a short thermal-style .avi. A central warm 'nostril' blob sits above
    a cool background; its temperature is a steady warm offset plus a smaller
    breathing oscillation at rr_brpm (warmer on exhale, cooler on inhale)."""
    rng = np.random.default_rng(seed)
    n = int(fps * dur)
    f = rr_brpm / 60.0
    w, h = size
    bg = 50.0                                    # cool background temperature
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    cy, cx = h / 2.0, w / 2.0
    # soft circular 'nostril' mask (Gaussian blob) at the centre
    blob = np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * (0.12 * w) ** 2)))
    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    vw = cv2.VideoWriter(path, fourcc, fps, size)
    for i in range(n):
        ti = i / fps
        breath = np.sin(2 * np.pi * f * ti) + 0.25 * np.sin(4 * np.pi * f * ti + 0.5)
        frame = bg + blob * (warm + amp * breath)
        frame = frame + noise * rng.standard_normal((h, w))
        gray = np.clip(frame, 0, 255).astype(np.uint8)
        vw.write(cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR))
    vw.release()
    return path


def extract_respiration_from_video(path, roi_frac=0.25, max_seconds=None):
    """Open a thermal video and return (signal, fps) where signal[i] is the mean
    intensity of a central ROI (the nostril patch) in frame i. A small central
    ROI isolates the airway from the static face background."""
    cap = cv2.VideoCapture(path)
    if not cap.isOpened():
        raise IOError(f"Could not open video: {path}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    max_frames = int(max_seconds * fps) if max_seconds else None
    means = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        g = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        h, w = g.shape[:2]
        rh, rw = int(h * roi_frac), int(w * roi_frac)
        y0, x0 = (h - rh) // 2, (w - rw) // 2
        roi = g[y0:y0 + rh, x0:x0 + rw]
        means.append(float(roi.mean()))
        if max_frames and len(means) >= max_frames:
            break
    cap.release()
    return np.asarray(means), float(fps)

In [ ]:
INJECTED_RR = 18.0
make_thermal_video("/tmp/thermal_nostril.avi", rr_brpm=INJECTED_RR, fps=30.0, dur=30.0)

sig, fps = extract_respiration_from_video("/tmp/thermal_nostril.avi", roi_frac=0.25)
print(f"Frames: {len(sig)}   fps: {fps:.1f}   duration: {len(sig) / fps:.1f} s")

rr_fft, freqs, power = rr_from_fft(sig, fps)
rr_pk, peaks, sig_f, ibi = rr_from_peaks(sig, fps)
print(f"True (injected) RR : {INJECTED_RR:.1f} breaths/min")
print(f"FFT estimate       : {rr_fft:.1f} breaths/min")
print(f"Peak estimate      : {rr_pk:.1f} breaths/min")
print(f"Signal quality     : {signal_quality(sig, fps):.2f}")

In [ ]:
# Grab one frame to show what the synthetic thermal image looks like.
cap = cv2.VideoCapture("/tmp/thermal_nostril.avi")
ok, frame0 = cap.read(); cap.release()
frame0 = cv2.cvtColor(frame0, cv2.COLOR_BGR2GRAY)

fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
ax[0].imshow(frame0, cmap="inferno")
ax[0].set_title("One thermal frame"); ax[0].axis("off")
h, w = frame0.shape
rh, rw = int(h * 0.25), int(w * 0.25)
ax[0].add_patch(plt.Rectangle(((w - rw) // 2, (h - rh) // 2), rw, rh,
                              ec="cyan", fc="none", lw=2))

tt = np.arange(len(sig)) / fps
ax[1].plot(tt, sig_f, color="#d62728", lw=1.2)
ax[1].plot(tt[peaks], sig_f[peaks], "kv", ms=6)
ax[1].set(title=f"ROI signal + breaths  →  {rr_pk:.1f} brpm",
          xlabel="time (s)", ylabel="a.u.")

brpm_axis = freqs * 60
m = (brpm_axis >= 0) & (brpm_axis <= 60)
ax[2].plot(brpm_axis[m], power[m] / power[m].max(), color="#2ca02c")
ax[2].axvline(rr_fft, color="r", ls="--", label=f"{rr_fft:.1f} brpm")
ax[2].set(title="Spectrum", xlabel="breaths/min", ylabel="norm. power")
ax[2].legend()
plt.tight_layout(); plt.show()

✅ The injected breathing rate comes back out of raw thermal pixels using the
identical `rr_from_fft` / `rr_from_peaks` code. That is the core JoulesEye
sensing step.

## 5. From respiration rate to Energy Expenditure (EE)

Why measure breathing at all? One powerful application is estimating **energy
expenditure** (calories burned), which for fitness and clinical monitoring is
driven by oxygen consumption. The physiology is the **Fick principle**:

> VO₂ = HR × SV × ΔavO₂        and        EE = VO₂ × γ

where **VO₂** is oxygen uptake (mL O₂/min), **HR** is heart rate (beats/min),
**SV** is stroke volume (mL blood/beat), **ΔavO₂** is the arterio-venous oxygen
difference (mL O₂ per mL blood), and **γ ≈ 5 kcal per litre of O₂** is the
caloric equivalent of oxygen.

A smartwatch measures **only HR**. But during exercise, SV and ΔavO₂ also rise —
and they differ between people (fitness, BMI) — so HR alone is a biased proxy for
VO₂. **Respiration rate tracks ventilation, which tracks VO₂ directly**, so
adding RR supplies the missing information.

Below we simulate a cohort, then compare an **HR-only** estimator against an
**RR + HR** estimator. The numbers reproduce the talk's headline: commercial
HR-only watches are around **37% off (MAPE)**, while JoulesEye's RR-informed
estimate is around **6%**, and it is more robust across BMI.

*(Everything here is clearly simulated and didactic — the point is the gap, not
the exact figures.)*

In [ ]:
GAMMA = 5.0   # kcal of energy released per litre of O2 consumed (caloric equivalent)

def fick_vo2(hr, sv_ml, davo2):
    """Fick principle: oxygen uptake VO2 (mL O2/min) = heart rate (beats/min)
    times stroke volume (mL blood/beat) times the arterio-venous O2 difference
    (mL O2 per mL blood)."""
    return hr * sv_ml * davo2

def energy_expenditure(vo2_ml_min, gamma=GAMMA):
    """Energy expenditure (kcal/min) = VO2 (converted to litres/min) times gamma."""
    return vo2_ml_min / 1000.0 * gamma

# ----- simulate a cohort of (participant, activity) samples -----
rng = np.random.default_rng(7)
N = 240
intensity = rng.uniform(0.0, 1.0, N)          # 0 = rest ... 1 = hard exercise
body = rng.normal(0.0, 1.0, N)                # between-subject fitness / BMI factor

HR    = 60.0 + 110.0 * intensity - 11.0 * body + rng.normal(0, 3, N)   # beats/min
SV    = np.clip(75.0 + 30.0 * intensity + 11.0 * body, 50, 130)        # mL/beat
dAVO2 = np.clip(0.05 + 0.10 * intensity + 0.013 * body, 0.03, 0.18)    # mL O2/mL blood

VO2 = fick_vo2(HR, SV, dAVO2)                  # mL O2/min
EE_true = energy_expenditure(VO2)              # kcal/min

# Respiration rate is driven by ventilation, which tracks VO2 -> RR is genuinely
# informative about metabolic rate (this is the JoulesEye insight).
RR = 10.0 + 13.0 * (VO2 / 1000.0) + rng.normal(0, 0.70, N)             # breaths/min

print(f"Cohort: {N} (participant, activity) samples")
print(f"HR range      : {HR.min():.0f} - {HR.max():.0f} beats/min")
print(f"RR range      : {RR.min():.0f} - {RR.max():.0f} breaths/min")
print(f"True EE range : {EE_true.min():.1f} - {EE_true.max():.1f} kcal/min")

In [ ]:
def mape(y_true, y_pred):
    """Mean absolute percentage error (%)."""
    return float(np.mean(np.abs(y_true - y_pred) / y_true) * 100.0)

def fit_linear(X, y):
    """Least-squares linear model with an intercept; returns predictions."""
    A = np.column_stack([np.ones(len(y)), X])
    coef, _, _, _ = np.linalg.lstsq(A, y, rcond=None)
    return A @ coef

# A smartwatch only sees HR -> calibrate the best possible HR-only mapping.
EE_hr_only = fit_linear(HR, EE_true)

# JoulesEye additionally measures RR from the thermal camera.
EE_rr_hr = fit_linear(np.column_stack([HR, RR]), EE_true)

mape_hr = mape(EE_true, EE_hr_only)
mape_rr = mape(EE_true, EE_rr_hr)

print("Energy-expenditure estimation error (MAPE, lower is better)")
print(f"  HR-only (smartwatch-style) : {mape_hr:.1f} %")
print(f"  RR + HR  (JoulesEye-style) : {mape_rr:.1f} %")
print(f"  Error reduction            : {(1 - mape_rr / mape_hr) * 100:.0f} %")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

labels = ["HR-only (smartwatch)", "RR + HR (JoulesEye)"]
vals = [mape_hr, mape_rr]
colors = ["#d62728", "#2ca02c"]
bars = ax[0].bar(labels, vals, color=colors, width=0.6)
for b, v in zip(bars, vals):
    ax[0].text(b.get_x() + b.get_width() / 2, v + 0.8, f"{v:.1f}%",
               ha="center", fontsize=12, fontweight="bold")
ax[0].set(title="Energy-expenditure error (MAPE)", ylabel="MAPE (%)")
ax[0].set_ylim(0, max(vals) * 1.3)

ax[1].plot([EE_true.min(), EE_true.max()], [EE_true.min(), EE_true.max()],
           "k--", lw=1, label="ideal")
ax[1].scatter(EE_true, EE_hr_only, s=10, alpha=0.5, color="#d62728", label="HR-only")
ax[1].scatter(EE_true, EE_rr_hr, s=10, alpha=0.5, color="#2ca02c", label="RR + HR")
ax[1].set(title="Predicted vs true EE", xlabel="true EE (kcal/min)",
          ylabel="estimated EE (kcal/min)")
ax[1].legend()
plt.tight_layout(); plt.show()

**Takeaway.** Heart rate alone cannot disambiguate metabolic states that share a
similar HR but differ in stroke volume and oxygen extraction. Respiration rate
adds an almost-direct readout of ventilation, collapsing the error from tens of
percent to single digits. That is the quantitative case for sensing the
neglected vital sign.

## 6. Bonus: why a fixed ROI fails under motion

Everything above assumed the nostril stayed in a fixed central box. During
exercise the head bobs and sways, and on low-texture thermal imagery classic
**optical flow** tends to fail — so a *fixed* ROI drifts off the nostril and
mixes in background, corrupting the signal. We simulate head sway and watch the
fixed-ROI estimate degrade. JoulesEye's fix is a learned **ROI localiser** that
keeps the box on the nostrils frame-to-frame.

In [ ]:
def make_thermal_video_moving(path, rr_brpm=18.0, fps=30.0, dur=30.0,
                              size=(160, 120), warm=120.0, amp=18.0,
                              sway_px=46.0, sway_hz=0.20, noise=3.0, seed=2):
    """Like make_thermal_video, but the warm nostril blob SWAYS across the frame
    (simulating head / body motion during exercise), so a fixed central ROI no
    longer stays on the nostril."""
    rng = np.random.default_rng(seed)
    n = int(fps * dur)
    f = rr_brpm / 60.0
    w, h = size
    bg = 50.0
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    fourcc = cv2.VideoWriter_fourcc(*"MJPG")
    vw = cv2.VideoWriter(path, fourcc, fps, size)
    for i in range(n):
        ti = i / fps
        cx = w / 2.0 + sway_px * np.sin(2 * np.pi * sway_hz * ti)
        cy = h / 2.0 + 0.4 * sway_px * np.sin(2 * np.pi * sway_hz * ti + 1.0)
        blob = np.exp(-(((xx - cx) ** 2 + (yy - cy) ** 2) / (2 * (0.12 * w) ** 2)))
        breath = np.sin(2 * np.pi * f * ti) + 0.25 * np.sin(4 * np.pi * f * ti + 0.5)
        frame = bg + blob * (warm + amp * breath)
        frame = frame + noise * rng.standard_normal((h, w))
        gray = np.clip(frame, 0, 255).astype(np.uint8)
        vw.write(cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR))
    vw.release()
    return path

In [ ]:
make_thermal_video_moving("/tmp/thermal_moving.avi", rr_brpm=INJECTED_RR, dur=30.0)
sig_m, fps_m = extract_respiration_from_video("/tmp/thermal_moving.avi", roi_frac=0.25)

rr_fft_m, _, _ = rr_from_fft(sig_m, fps_m)
rr_pk_m, _, sig_mf, _ = rr_from_peaks(sig_m, fps_m)
q_static = signal_quality(sig, fps)        # the no-motion video from section 4
q_moving = signal_quality(sig_m, fps_m)

err_static = abs(rr_fft - INJECTED_RR)
err_moving = abs(rr_fft_m - INJECTED_RR)
print(f"Injected RR                          : {INJECTED_RR:.1f} brpm")
print(f"Fixed ROI, NO motion   -> RR (FFT)   : {rr_fft:.1f} brpm   (error {err_static:.1f}, quality {q_static:.2f})")
print(f"Fixed ROI, WITH motion -> RR (FFT)   : {rr_fft_m:.1f} brpm   (error {err_moving:.1f}, quality {q_moving:.2f})")
print("Under motion the fixed ROI locks onto the head-sway frequency, not breathing;")
print("this is exactly why JoulesEye needs an ROI localiser to track the nostril.")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))

tt = np.arange(len(sig)) / fps
ax[0].plot(tt, detrend(sig), color="#2ca02c", lw=1.1)
ax[0].set(title=f"Fixed ROI, no motion (quality {q_static:.2f})",
          xlabel="time (s)", ylabel="ROI mean (detrended)")

ttm = np.arange(len(sig_m)) / fps_m
ax[1].plot(ttm, detrend(sig_m), color="#d62728", lw=1.1)
ax[1].set(title=f"Fixed ROI, with motion (quality {q_moving:.2f})",
          xlabel="time (s)", ylabel="ROI mean (detrended)")
plt.tight_layout(); plt.show()

As the warm nostril blob sways in and out of the fixed box, the ROI mean is
dominated by a spurious component at the head-sway frequency rather than at the
breathing rate. The FFT therefore locks onto the wrong peak and returns a
**biased respiration rate** (notice the static estimate is correct, the moving
one is not). A tracker that follows the nostril restores a clean trace — that is
the job of the ROI localiser.

## 🔬 Bonus: real thermal data from the lab (ApneaEye)

Everything above used synthetic data so the true answer was known. Now we run
the **same respiration pipeline on a real thermal recording** released by the
group — a ~72-second clip from the ApneaEye project
([github.com/AyushShrivstava/ApneaEye_Demo](https://github.com/AyushShrivstava/ApneaEye_Demo)).

A real thermal frame does not tell us *where* the nostril is. The talk's
**ROI localiser** solves this; here we use a simple, transparent version of the
same idea: **scan a grid of candidate patches and keep the one whose intensity
oscillates most strongly in the breathing band (0.1–0.6 Hz)** — that patch is
the nostril. The cell downloads the clip automatically in Colab and is skipped
gracefully if you are offline.

In [ ]:
import urllib.request, os
THERMAL_URL = "https://raw.githubusercontent.com/AyushShrivstava/ApneaEye_Demo/main/Data/demo_ApneaEye_2025-11-12_01-18-18.avi"
thermal_path = "/tmp/demo_apnea_thermal.avi"
have_thermal = False
try:
    if not os.path.exists(thermal_path) or os.path.getsize(thermal_path) < 10000:
        print("Downloading real thermal clip (~15 MB) ...")
        urllib.request.urlretrieve(THERMAL_URL, thermal_path)
    have_thermal = os.path.getsize(thermal_path) > 10000
    print("Ready:", thermal_path, "(%d bytes)" % os.path.getsize(thermal_path))
except Exception as e:
    print("Could not fetch real data (offline?). Skipping real-data demo.")
    print(" ", e)

In [ ]:
from scipy.signal import detrend, butter, filtfilt, find_peaks
from matplotlib.patches import Rectangle

if have_thermal:
    cap = cv2.VideoCapture(thermal_path)
    fps_r = cap.get(cv2.CAP_PROP_FPS) or 25.0
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok:
            break
        frames.append(fr.mean(axis=2))          # grayscale ~ temperature proxy
    cap.release()
    vid = np.asarray(frames)
    T, Hh, Ww = vid.shape
    print("Real thermal video: %dx%d, %d frames @ %.0f fps = %.0f s"
          % (Ww, Hh, T, fps_r, T / fps_r))

    def band_info(sig, fs, lo=0.1, hi=0.6):
        x = detrend(sig); N = len(x)
        f = np.fft.rfftfreq(N, 1.0 / fs)
        P = np.abs(np.fft.rfft(x * np.hanning(N))) ** 2
        b = (f >= lo) & (f <= hi)
        return P[b].sum() / P.sum(), f[b][np.argmax(P[b])]

    # --- localise the nostril: most breathing-band power over a grid of patches
    best = None
    rh, rw = Hh // 8, Ww // 10
    for fy in np.linspace(0.25, 0.75, 6):
        for fx in np.linspace(0.30, 0.70, 6):
            cy, cx = int(Hh * fy), int(Ww * fx)
            patch = vid[:, max(0, cy - rh):cy + rh, max(0, cx - rw):cx + rw].mean(axis=(1, 2))
            frac, fpk = band_info(patch, fps_r)
            if best is None or frac > best[0]:
                best = (frac, cy, cx, patch, fpk)
    frac, cy, cx, roi_sig, fpk = best
    rr_fft = fpk * 60.0

    ny = fps_r / 2.0
    bb, aa = butter(3, [0.1 / ny, min(0.6, 0.99 * ny) / ny], btype="band")
    roi_f = filtfilt(bb, aa, detrend(roi_sig))
    pks, _ = find_peaks(roi_f, distance=int(fps_r / 0.6))
    rr_pk = 60.0 / np.median(np.diff(pks) / fps_r) if len(pks) > 1 else float("nan")

    print("Localised nostril ROI  : breathing-band power fraction = %.2f" % frac)
    print("Respiration rate (FFT)  : %.1f breaths/min" % rr_fft)
    print("Respiration rate (peaks): %.1f breaths/min" % rr_pk)

    fig, ax = plt.subplots(1, 3, figsize=(13, 3.4))
    ax[0].imshow(vid.mean(0), cmap="inferno")
    ax[0].add_patch(Rectangle((cx - rw, cy - rh), 2 * rw, 2 * rh,
                              ec="cyan", fc="none", lw=2))
    ax[0].set_title("Real thermal frame + localised ROI"); ax[0].axis("off")
    tt = np.arange(len(roi_f)) / fps_r
    ax[1].plot(tt, roi_f, color="#d62728", lw=1)
    ax[1].plot(tt[pks], roi_f[pks], "kv", ms=5)
    ax[1].set_title("ROI breathing signal  ->  %.1f brpm" % rr_pk)
    ax[1].set_xlabel("time (s)")
    x = detrend(roi_sig); N = len(x)
    f = np.fft.rfftfreq(N, 1.0 / fps_r); P = np.abs(np.fft.rfft(x * np.hanning(N))) ** 2
    m = (f * 60 >= 5) & (f * 60 <= 40)
    ax[2].plot(f[m] * 60, P[m] / P[m].max(), color="#2ca02c")
    ax[2].axvline(rr_fft, color="r", ls="--", label="%.1f brpm" % rr_fft)
    ax[2].set_title("Spectrum"); ax[2].set_xlabel("breaths/min"); ax[2].legend()
    plt.tight_layout(); plt.show()
    print("\nThis is a REAL person breathing, measured from a thermal camera — "
          "no synthetic data. The ROI scan is a minimal stand-in for JoulesEye/"
          "ApneaEye's learned ROI localiser.")

## 7. Limitations, exercises, and safety

**Why it works.** Exhaled air is warm and inhaled air is cool, so the nostril
region is *temperature-modulated at the breathing rate*. Averaging an ROI gives a
high-SNR 1-D signal, and breathing sits in a narrow, low-frequency band that is
easy to isolate.

**Where it fails**
- *Motion* (exercise, talking) moves the nostril out of a fixed ROI — hence the
  need for an ROI localiser.
- *Low frame rate or short clips* limit frequency resolution (1/duration in Hz).
- *Mouth breathing, masks, cold ambient air* reduce the temperature contrast.
- *Thermal cameras* remain relatively expensive and need calibration.

**Exercises**
1. Change `INJECTED_RR` to 9, 24 and 33 brpm and confirm the pipeline tracks it.
   Where does the FFT resolution start to hurt, and how does a longer `dur` help?
2. Replace the FFT peak-pick with **parabolic interpolation** around the peak for
   sub-bin resolution. How much does the estimate change at short durations?
3. Add a **sliding window** (e.g. 20 s, 1 s hop) to plot respiration rate *over
   time*, then make the synthetic rate ramp up and watch the estimate follow.
4. In the EE section, widen the `body` (BMI / fitness) spread and plot how the
   HR-only error grows while the RR + HR error stays low — reproducing the
   "robust across BMI" claim.

**Safety.** Educational only. Not validated, not a medical device. Do not use it
for any diagnostic or clinical decision.

---
*Made for the ACM Summer School on AI for Health.*